# E4: the frame probabilities of the CTC models, for decoding with language models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivannamor/multilingual-asr-lab/blob/main/notebooks/e4_logprobs_colab.ipynb)

Experiment E4 of [RESULTS.md](https://github.com/ivannamor/multilingual-asr-lab/blob/main/RESULTS.md)
decodes the CTC models of E1b with n-gram language models. Only the acoustic model needs a GPU: this
notebook saves its log probabilities for the dev and test sets, and all the decoding runs afterwards
on a CPU, without touching the audio again.

1. On Google Colab: *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all* and keep the
   tab open. The first run installs the package and restarts the session: then *Run all* again.
   Colab reports this as a crash "for an unknown reason": it is expected.
2. At the end the browser downloads `e4_logprobs.zip` (if it does not, download it from the *Files*
   panel): log probabilities and nothing else, no audio and no weights.
3. In a local clone, from the repository root: `python3 -m zipfile -e e4_logprobs.zip .`

If the session is interrupted but the machine survives, run all cells again: a finished file is
kept. A new machine starts from zero, because the files are on its local disk.


## 1. Clone and install

In [ ]:
import importlib.util
import os
import subprocess
import sys
import time
from pathlib import Path

# Work inside a clone of the repository, so that every run.json records the git commit and the
# run directories end up where the experiment scripts expect them.
if Path("../pyproject.toml").exists():  # started from notebooks/ in a local clone
    os.chdir("..")
elif not Path("pyproject.toml").exists():  # on Colab
    if not Path("multilingual-asr-lab").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ivannamor/multilingual-asr-lab"],
            check=True,
        )
    os.chdir("multilingual-asr-lab")
if importlib.util.find_spec("asr_lab") is None:
    # The hf extra adds librosa and, if Colab's is too old, a newer Transformers
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[hf]"], check=True)
    # pip may have upgraded packages that the kernel has already imported (numpy), and mixing two
    # versions in one process fails later: restart the session, then run all the cells again
    # (this one will skip the installation)
    # Colab reports this as a crash "for an unknown reason": it is expected. The pause lets the
    # message reach the notebook before the kernel dies.
    print("Installed: the session restarts. Run all the cells again.", flush=True)
    time.sleep(2)
    os.kill(os.getpid(), 9)
print(Path.cwd())

## 2. Settings


In [ ]:
SPEC_PATH = Path("experiments/E4/lm.toml")
OUTPUT = Path("data/e4_logprobs")

## 3. Data

The dev and test sets of FLEURS and VoxPopuli: about 4 GB, as the local scripts download them.


In [ ]:
import torch

from asr_lab.data.ingest import ingest_fleurs
from asr_lab.data.voxpopuli import ingest_voxpopuli, load_data_spec
from asr_lab.lm.corpus import load_spec

if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU")

spec = load_spec(SPEC_PATH)
voxpopuli = {"dev": "experiments/E4/voxpopuli_dev.toml", "test": "experiments/E7/test.toml"}
for test_set in spec["test_sets"]:
    for split in ("dev", "test"):
        manifest = Path(test_set[split])
        if not manifest.exists():
            if test_set["name"].startswith("fleurs"):
                ingest_fleurs(test_set["lang"], split, manifest.parent)
            else:
                ingest_voxpopuli(load_data_spec(Path(voxpopuli[split])))
        print(manifest)

## 4. Log probabilities

One checkpoint per language, as pinned for E1b, and one file per split.


In [ ]:
from asr_lab.lm.experiment import run_lm_logprobs

written = run_lm_logprobs(SPEC_PATH, OUTPUT, "cuda")
for path in written:
    print(f"{path}: {path.stat().st_size / 1e6:.0f} MB")

## 5. Download the results


In [ ]:
import zipfile

archive = Path("e4_logprobs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_STORED) as zipped:  # already compressed
    for path in written:
        zipped.write(path, path.as_posix())
print(f"{archive.resolve()}: {archive.stat().st_size / 1e6:.1f} MB")

try:
    from google.colab import files

    files.download(str(archive))
except ImportError:  # not on Colab
    pass